# H&M Article Text Embedding Extraction

## 1. Objective and Artifact Contract

This notebook extracts static text embeddings for every valid article in
`articles.csv` using `sentence-transformers/all-MiniLM-L6-v2`. Each article
receives one L2-normalized, 384-dimensional FP32 vector. The model runs in
inference mode with its pretrained pooling and native token limit.

Only article metadata is required. Purchase history, customer data, temporal
splits, training objectives and recommendation metrics are outside this task.
The metadata is a static snapshot; its historical availability is not known.

| Artifact | Contract |
| --- | --- |
| `text_embeddings.npy` | FP32 matrix with shape `(N, 384)` |
| `text_article_ids.parquet` | Sorted IDs, row positions, full text and token audit |
| `text_faiss.index` | Exact inner-product index aligned with matrix rows |
| `text_features_manifest.json` | Source/model identities, checksums and measurements |

`N` is the validated article count. The explicit task requires a self-contained
Colab notebook and the `original data/text_features/` destination. These choices
are retained as task-specific exceptions to the general repository layout.

**Execution status:** local synthetic checks validate preprocessing and artifact
handling. Full extraction on Colab L4 has not been verified. Run cells in order
from a fresh GPU runtime; later sections report measurements from that run.

### Contents

1. [Objective and artifact contract](#1-objective-and-artifact-contract)
2. [Dependencies and Drive access](#2-dependencies-and-drive-access)
3. [Configuration and runtime](#3-configuration-and-runtime)
4. [Source staging and article validation](#4-source-staging-and-article-validation)
5. [Text construction and inspection](#5-text-construction-and-inspection)
6. [Model revision and token audit](#6-model-revision-and-token-audit)
7. [GPU extraction and measurements](#7-gpu-extraction-and-measurements)
8. [Index construction and local validation](#8-index-construction-and-local-validation)
9. [Verified artifact publication](#9-verified-artifact-publication)
10. [Results, reload and limitations](#10-results-reload-and-limitations)

## 2. Dependencies and Drive Access

Install the extraction dependencies while constraining PyTorch packages to the
versions already installed by Colab. This preserves the native CUDA environment.
Mount Drive to read the source article table and publish validated outputs.

In [ ]:
import importlib.metadata
import subprocess
import sys
import tempfile

constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
if not any(p.startswith("torch==") for p in constraints):
    raise RuntimeError("Select a Colab GPU runtime with native CUDA PyTorch.")
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as constraint_file:
    constraint_file.write("\n".join(constraints))
    constraint_file.flush()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--constraint",
            constraint_file.name,
            "sentence-transformers",
            "faiss-cpu",
            "pyarrow",
            "tqdm",
        ],
        check=True,
    )

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configuration and Runtime

The configuration makes input fields, model settings and output paths explicit.
`article_id` is an identifier, not an input feature. `TEXT_COLUMNS` defines the
14 descriptive fields in their required order.

The target runtime is a Colab L4 GPU. Batch size 256 is an initial setting whose
suitability must be measured. CUDA is required, and TF32 is disabled to keep the
baseline computation in FP32. Seeds are recorded for reproducibility; they do
not guarantee identical results across hardware or library versions.

In [ ]:
import csv
import hashlib
import json
import math
import re
import resource
import shutil
import time
from pathlib import Path

import faiss
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE_DIR = Path(
    "/content/drive/MyDrive/projects/e_commerce_recsys/data/original data"
)
ARTICLE_PATH = SOURCE_DIR / "articles.csv"
OUTPUT_DIR = SOURCE_DIR / "text_features"
LOCAL_WORK_DIR = Path("/content/hm_text_embeddings")
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = (
    None  # Optional immutable commit SHA; otherwise resolve once per run.
)
EMBEDDING_DIM = 384
BATCH_SIZE = 256
TEXT_COLUMNS = [
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_code",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name",
    "detail_desc",
]
SEPARATOR = "|"
PREPROCESSING = {
    "version": "hm-text-values-pipe-14-v1",
    "rules": (
        "strip; collapse whitespace; skip actual null/empty; retain literal categories"
    ),
    "columns": TEXT_COLUMNS,
    "separator": SEPARATOR,
}
MAPPING_SCHEMA = pa.schema(
    [
        ("embedding_row", pa.int64()),
        ("article_id", pa.string()),
        ("text", pa.string()),
        ("token_count", pa.int32()),
        ("was_truncated", pa.bool_()),
    ]
)
ARTIFACTS = (
    "text_embeddings.npy",
    "text_article_ids.parquet",
    "text_faiss.index",
)
MANIFEST_NAME = "text_features_manifest.json"

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA unavailable. Select a Colab GPU runtime; extraction stopped."
    )
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
torch.cuda.reset_peak_memory_stats()
free_vram, total_vram = torch.cuda.mem_get_info()
print(
    {
        "gpu": torch.cuda.get_device_name(),
        "free_GPU_GiB": free_vram / 1024**3,
        "total_GPU_GiB": total_vram / 1024**3,
        "starting_batch_size": BATCH_SIZE,
    }
)

## 4. Source Staging and Article Validation

Stage `articles.csv` once onto local disk so extraction does not repeatedly read
Drive. Compare the staged file with the source checksum before parsing it.

The CSV reader retains strings without inferring missing-value categories.
Empty fields stay empty, while literal values such as `None` and `Unknown`
remain available to text construction. Reject invalid or duplicate article IDs
and articles without usable text. Sort the validated IDs before assigning rows.

In [ ]:
def require(condition: bool, message: str) -> None:
    """Raise ValueError when a required input or artifact condition fails."""
    if not condition:
        raise ValueError(message)


def sha256_file(path: Path) -> str:
    """Compute a file's SHA-256 using bounded reads.

    Args:
        path: File to read without modifying it.

    Returns:
        Hexadecimal SHA-256 digest of the complete file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def joined_text(row: dict[str, object]) -> str:
    """Join descriptive values in the configured field order.

    Args:
        row: Article fields. None, floating-point NaN and blank values are
            skipped; literal category strings and repeated values are retained.

    Returns:
        Cleaned values separated by literal pipes, without feature labels.
        Returns an empty string when no field contains usable text."""
    values = []
    for column in TEXT_COLUMNS:
        value = row.get(column)
        if value is None or (isinstance(value, float) and math.isnan(value)):
            continue
        value = re.sub(r"\s+", " ", str(value)).strip()
        if value:
            values.append(value)
    return SEPARATOR.join(values)

In [ ]:
def read_articles(path: Path) -> tuple[list[str], list[str]]:
    """Read all articles and construct aligned IDs and input texts.

    CSV values remain strings, preserving leading zeros and literal categories.
    No rows are filtered by purchase activity or deduplicated by text.

    Args:
        path: Source CSV containing article_id and every required text field.

    Returns:
        Two aligned lists: unique IDs in ascending order and nonempty texts.

    Raises:
        ValueError: Required columns are absent, the table is empty, an ID is
            invalid or duplicated, or an article has no usable text."""
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        require(
            set(["article_id", *TEXT_COLUMNS]) <= set(reader.fieldnames or []),
            "Missing article_id or required text columns.",
        )
        records = []
        seen = set()
        for line, row in enumerate(reader, 2):
            item = row.get("article_id", "")
            require(
                isinstance(item, str)
                and re.fullmatch(r"[0-9]{10}", item) is not None,
                f"Invalid ten-digit article_id at CSV line {line}.",
            )
            require(
                item not in seen, f"Duplicate article_id at CSV line {line}."
            )
            text = joined_text(row)
            require(
                bool(text), f"Article {item} has no usable descriptive text."
            )
            seen.add(item)
            records.append((item, text))
    require(bool(records), "Article table is empty.")
    records.sort()
    return [r[0] for r in records], [r[1] for r in records]

In [ ]:
def verify_files(directory: Path, manifest: dict) -> None:
    """Verify the three payload files against a completed manifest.

    Args:
        directory: Directory containing the named payload files.
        manifest: Completed manifest with expected filenames, sizes and hashes.

    Raises:
        ValueError: Completion status, filenames, sizes or checksums differ."""
    require(
        manifest.get("validation_status") == "validated",
        "Manifest is incomplete.",
    )
    require(
        set(manifest["files"]) == set(ARTIFACTS),
        "Unexpected artifact file contract.",
    )
    for name, info in manifest["files"].items():
        path = directory / name
        require(
            path.is_file() and path.stat().st_size == info["bytes"],
            f"Size mismatch: {name}",
        )
        require(
            sha256_file(path) == info["sha256"], f"Checksum mismatch: {name}"
        )

In [ ]:
require(ARTICLE_PATH.is_file(), f"Missing source: {ARTICLE_PATH}")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=LOCAL_WORK_DIR))
require(
    shutil.disk_usage(RUN_DIR).free > ARTICLE_PATH.stat().st_size * 2 + 1024**3,
    "Insufficient local staging space.",
)
source_sha256 = sha256_file(ARTICLE_PATH)
local_articles = RUN_DIR / "articles.csv"
shutil.copyfile(ARTICLE_PATH, local_articles)
require(
    sha256_file(local_articles) == source_sha256,
    "Staged article checksum mismatch.",
)
article_ids, texts = read_articles(local_articles)
N = len(article_ids)
print({"articles": N, "included_text_columns": TEXT_COLUMNS})

## 5. Text Construction and Inspection

Join cleaned, nonempty field values with the literal separator `|`. Preserve
field order and repeated values. Feature names, numeric category codes and
article IDs are excluded from the text. Distinct articles with identical text
remain separate rows.

Inspect the sample below for field order, whitespace handling and unintended
labels before starting encoding. The displayed strings are the full inputs,
before tokenizer truncation.

In [ ]:
for index in range(min(5, N)):
    print({"article_id": article_ids[index], "text": texts[index]})

## 6. Model Revision and Token Audit

Resolve an immutable model revision and compare the source and preprocessing
contract with any existing publication. Matching outputs are staged and
verified for reuse. Incomplete or incompatible outputs stop the run without
replacing files.

For a new extraction, load the model's native pooling configuration and measure
untruncated token lengths in bounded batches, including special tokens. The
p50, p95, p99 and maximum lengths describe the input distribution. The count and
percentage above `max_seq_length` quantify truncation at the native token limit.
The full text and a per-article truncation flag remain in the audit mapping.

In [ ]:
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

revision = MODEL_REVISION or HfApi().model_info(MODEL_ID).sha
require(
    isinstance(revision, str)
    and re.fullmatch(r"[0-9a-fA-F]{40}", revision) is not None,
    "An immutable model commit SHA is required.",
)
contract = {
    "model_id": MODEL_ID,
    "model_revision": revision,
    "preprocessing": PREPROCESSING,
    "source_sha256": source_sha256,
    "article_count": N,
    "embedding_dim": EMBEDDING_DIM,
    "compute_precision": "float32",
    "normalize_embeddings": True,
}
manifest_path = OUTPUT_DIR / MANIFEST_NAME
REUSED = OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir())
if REUSED:
    require(
        manifest_path.is_file(),
        "Existing output is incomplete; inspect it manually. No files overwritten.",
    )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    require(
        manifest.get("contract") == contract,
        "Existing output contract differs; no files overwritten.",
    )
    verify_files(OUTPUT_DIR, manifest)
    for name in ARTIFACTS:
        shutil.copyfile(OUTPUT_DIR / name, RUN_DIR / name)
    verify_files(RUN_DIR, manifest)
    max_seq_length = manifest["max_seq_length"]
    print({"reused_revision": revision, "token_audit": manifest["token_audit"]})
else:
    model = SentenceTransformer(MODEL_ID, revision=revision, device="cuda")
    model.float().eval()
    require(
        model.get_sentence_embedding_dimension() == EMBEDDING_DIM,
        "Unexpected embedding dimension.",
    )
    max_seq_length = model.max_seq_length
    require(
        isinstance(max_seq_length, int) and max_seq_length > 0,
        "Invalid native sequence limit.",
    )
    token_counts = np.empty(N, dtype=np.int32)
    for start in range(0, N, 512):
        encoded = model.tokenizer(
            texts[start : start + 512],
            truncation=False,
            padding=False,
            add_special_tokens=True,
        )
        token_counts[start : start + len(encoded["input_ids"])] = [
            len(x) for x in encoded["input_ids"]
        ]
    truncated = token_counts > max_seq_length
    token_audit = {
        **dict(
            zip(
                ["p50", "p95", "p99"],
                np.percentile(token_counts, [50, 95, 99]).tolist(),
                strict=True,
            )
        ),
        "maximum": int(token_counts.max()),
        "max_seq_length": max_seq_length,
        "truncated_count": int(truncated.sum()),
        "truncated_rate": float(truncated.mean()),
    }
    print(token_audit)
    print({"truncated_percent": 100 * token_audit["truncated_rate"]})
    if truncated.any():
        print(
            "WARNING: All fields enter the source text; tokens beyond the native "
            "model limit are truncated during encoding."
        )
        print(
            [
                {"article_id": article_ids[i], "tokens": int(token_counts[i])}
                for i in np.flatnonzero(truncated)[:10]
            ]
        )

### Interpreting the Token Audit

An over-limit input includes all 14 fields in its source string, but the model
encodes only tokens within its native window. Review the reported rate and
sample IDs to understand how often trailing content is omitted. The notebook
retains the native limit and does not introduce chunked encoding.

## 7. GPU Extraction and Measurements

Encode articles in bounded blocks using FP32 inference, native pooling and an
empty prompt. A CUDA out-of-memory error reduces the batch size and retries the
same unfinished block. Other exceptions propagate; no article is skipped.

Write vectors into an `(N, 384)` memory-mapped array in article order. CUDA
synchronization brackets the encoding timer. The progress bar and final
throughput use observed work rather than a preset runtime estimate.

In [ ]:
from tqdm.auto import tqdm

if not REUSED:
    require(
        shutil.disk_usage(RUN_DIR).free > N * EMBEDDING_DIM * 4 * 4 + 1024**3,
        "Insufficient local space for embeddings, index and validation.",
    )
    vectors = np.lib.format.open_memmap(
        RUN_DIR / "text_embeddings.npy",
        mode="w+",
        dtype=np.float32,
        shape=(N, EMBEDDING_DIM),
    )
    actual_batch_size = BATCH_SIZE
    batch_sizes_used = set()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with tqdm(total=N, unit="article", desc="Text encoding") as progress:
        for start in range(0, N, 2048):
            block = texts[start : start + 2048]
            while True:
                try:
                    with torch.inference_mode():
                        result = model.encode(
                            block,
                            batch_size=actual_batch_size,
                            device="cuda",
                            convert_to_numpy=True,
                            normalize_embeddings=True,
                            precision="float32",
                            prompt="",
                            truncate_dim=None,
                            show_progress_bar=False,
                        )
                    break
                except torch.cuda.OutOfMemoryError:
                    if actual_batch_size == 1:
                        raise
                    actual_batch_size = max(1, actual_batch_size // 2)
                    torch.cuda.empty_cache()
                    print(
                        "CUDA OOM: retrying unfinished block with batch size "
                        f"{actual_batch_size}."
                    )
            vectors[start : start + len(block)] = result
            batch_sizes_used.add(actual_batch_size)
            progress.update(len(block))
    vectors.flush()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    measured = {
        "encoding_seconds": elapsed,
        "articles_per_second": N / elapsed,
        "articles_encoded": N,
        "peak_process_RAM_GiB": resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss
        / 1024**2,
        "peak_GPU_allocated_GiB": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_GPU_reserved_GiB": torch.cuda.max_memory_reserved() / 1024**3,
    }
    print(
        {**measured, "shape": list(vectors.shape), "dtype": str(vectors.dtype)}
    )
    mapping = pa.Table.from_arrays(
        [
            pa.array(np.arange(N, dtype=np.int64)),
            pa.array(article_ids),
            pa.array(texts),
            pa.array(token_counts),
            pa.array(truncated),
        ],
        schema=MAPPING_SCHEMA,
    )
    pq.write_table(
        mapping, RUN_DIR / "text_article_ids.parquet", compression="zstd"
    )

### Interpreting the Encoding Measurements

For a new extraction, the encoded count must equal the validated article count.
Throughput measures encoding time; it excludes installation, staging and Drive
publication. Peak process RAM includes earlier work in this runtime. GPU peaks
are measured since the reset in the runtime check. These measurements describe
this run and do not establish that batch size 256 is optimal.

## 8. Index Construction and Local Validation

Build an exact CPU `IndexFlatIP` over the normalized FP32 vectors. Inner products
between these normalized vectors represent cosine similarity.

Reload the matrix, mapping and index to check dimensions, dtypes, finite values,
unit norms, unique IDs and contiguous row positions. Reconstructed FAISS vectors
must match their matrix rows. Similarity checks allow tied neighbors because
different articles may legitimately have identical text and embeddings.

Write the local manifest only after these checks pass. Its payload checksums
cover the three data files; the manifest does not checksum itself.

In [ ]:
def validate_artifacts(
    directory: Path, ids: list[str], full_texts: list[str], limit: int
) -> dict:
    """Reload artifacts and validate their article-to-vector alignment.

    Duplicate vectors are allowed. Validation checks reconstruction alignment
    and similarity values rather than requiring unique self-matches.

    Args:
        directory: Local directory containing the three payload files.
        ids: Sorted, unique source article IDs in expected matrix-row order.
        full_texts: Full pre-truncation texts in the same order as ids.
        limit: Native model token limit used to derive truncation flags.

    Returns:
        Validated matrix shape, mapping row count and completion status.

    Raises:
        ValueError: A shape, dtype, normalization, mapping, token-audit or
            FAISS alignment check fails."""
    matrix = np.load(
        directory / "text_embeddings.npy", mmap_mode="r", allow_pickle=False
    )
    require(
        matrix.shape == (len(ids), EMBEDDING_DIM)
        and matrix.dtype == np.float32,
        "Embedding shape/dtype mismatch.",
    )
    for start in range(0, len(ids), 4096):
        block = matrix[start : start + 4096]
        norms = np.linalg.norm(block, axis=1)
        require(
            np.isfinite(block).all()
            and np.all(norms > 0)
            and np.allclose(norms, 1, atol=0.002),
            "Nonfinite, zero or non-unit embedding.",
        )
    table = pq.read_table(directory / "text_article_ids.parquet")
    require(
        table.schema == MAPPING_SCHEMA and len(table) == len(ids),
        "Mapping schema/row mismatch.",
    )
    require(
        all(column.null_count == 0 for column in table.columns),
        "Null mapping values.",
    )
    require(
        len(set(ids)) == len(ids) and table["article_id"].to_pylist() == ids,
        "Article mapping mismatch.",
    )
    require(
        table["embedding_row"].to_pylist() == list(range(len(ids))),
        "Noncontiguous mapping rows.",
    )
    require(
        table["text"].to_pylist() == full_texts, "Full text audit mismatch."
    )
    counts = table["token_count"].to_numpy()
    require(
        np.all(counts > 0)
        and np.array_equal(table["was_truncated"].to_numpy(), counts > limit),
        "Token/truncation audit mismatch.",
    )
    index = faiss.read_index(str(directory / "text_faiss.index"))
    require(
        isinstance(index, faiss.IndexFlatIP)
        and index.d == EMBEDDING_DIM
        and index.ntotal == len(ids),
        "FAISS type/dimension/count mismatch.",
    )
    sample = np.unique(
        np.linspace(0, len(ids) - 1, min(64, len(ids)), dtype=np.int64)
    )
    for row in sample:
        require(
            np.array_equal(index.reconstruct(int(row)), matrix[row]),
            "FAISS row alignment mismatch.",
        )
    scores, positions = index.search(np.ascontiguousarray(matrix[sample]), 1)
    for query, score, position in zip(
        matrix[sample], scores[:, 0], positions[:, 0], strict=True
    ):
        require(
            0 <= position < len(ids)
            and np.isclose(score, np.dot(query, matrix[position]), atol=2e-5),
            "FAISS similarity mismatch.",
        )
    return {
        "shape": list(matrix.shape),
        "mapping_rows": len(table),
        "validation_status": "validated",
    }

In [ ]:
if not REUSED:
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    for start in range(0, N, 4096):
        index.add(
            np.ascontiguousarray(
                vectors[start : start + 4096], dtype=np.float32
            )
        )
    faiss.write_index(index, str(RUN_DIR / "text_faiss.index"))
validation = validate_artifacts(RUN_DIR, article_ids, texts, max_seq_length)
print(validation)

In [ ]:
if not REUSED:
    versions = {
        name: importlib.metadata.version(name)
        for name in (
            "torch",
            "sentence-transformers",
            "transformers",
            "huggingface-hub",
            "numpy",
            "pyarrow",
            "faiss-cpu",
            "tqdm",
        )
    }
    manifest = {
        "contract": contract,
        "max_seq_length": max_seq_length,
        "token_audit": token_audit,
        "embedding_shape": [N, EMBEDDING_DIM],
        "embedding_dtype": "float32",
        "normalization": "L2",
        "index_type": "IndexFlatIP",
        "index_dimension": EMBEDDING_DIM,
        "index_ntotal": N,
        "batch_size_initial": BATCH_SIZE,
        "batch_size_final": actual_batch_size,
        "batch_sizes_used": sorted(batch_sizes_used),
        "versions": versions,
        "runtime": {
            "python": sys.version,
            "gpu": torch.cuda.get_device_name(),
            "torch_cuda": torch.version.cuda,
            "seed": 42,
        },
        "measurements": measured,
        "files": {
            name: {
                "bytes": (RUN_DIR / name).stat().st_size,
                "sha256": sha256_file(RUN_DIR / name),
            }
            for name in ARTIFACTS
        },
        "validation_status": "validated",
    }
    (RUN_DIR / MANIFEST_NAME).write_text(
        json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
    )

## 9. Verified Artifact Publication

Copy the validated payload to the requested Drive directory and verify its
sizes and SHA-256 checksums. Publish the completion manifest last so a partial
copy cannot be mistaken for a completed extraction.

Exclusive file creation prevents replacement of existing files. A failed copy
leaves an incomplete destination that requires manual review. A matching,
completed publication is verified and reused.

In [ ]:
def publish_verified(local: Path, destination: Path, manifest: dict) -> None:
    """Publish verified artifacts without replacing existing files.

    Verifies local checksums, copies the payload, verifies destination checksums
    and writes the completion manifest last. Failed copies remain incomplete.

    Args:
        local: Directory containing the validated payload files.
        destination: Empty output directory, created if absent.
        manifest: Completed local manifest describing the payload.

    Raises:
        ValueError: Integrity checks fail, the destination is nonempty, or
            available destination space is insufficient.
        OSError: File creation or copying fails, including a concurrent write."""
    verify_files(local, manifest)
    destination.mkdir(parents=True, exist_ok=True)
    require(
        not any(destination.iterdir()),
        "Destination is nonempty; publication stopped.",
    )
    require(
        shutil.disk_usage(destination).free
        > sum(v["bytes"] for v in manifest["files"].values()) + 1024**2,
        "Insufficient destination space.",
    )
    for name in ARTIFACTS:
        with (
            (local / name).open("rb") as source,
            (destination / name).open("xb") as target,
        ):
            shutil.copyfileobj(source, target, length=8 * 1024**2)
    verify_files(destination, manifest)
    with (destination / MANIFEST_NAME).open("x", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, sort_keys=True)
    require(
        json.loads((destination / MANIFEST_NAME).read_text(encoding="utf-8"))
        == manifest,
        "Published manifest verification failed.",
    )

In [ ]:
if not REUSED:
    publish_verified(RUN_DIR, OUTPUT_DIR, manifest)
else:
    verify_files(OUTPUT_DIR, manifest)
print(
    "Verified completed artifacts reused."
    if REUSED
    else "Validated artifacts published; manifest written last."
)

## 10. Results, Reload and Limitations

The output table reports artifact paths, sizes, dimensions or row counts, and
validation status. The recorded measurements describe encoding, including
throughput and peak memory. When outputs are reused, those measurements belong
to the original extraction; they are not timings for the current session.

The reload example uses local, checksum-verified files and repeats structural
validation. Downstream consumers must use `embedding_row` to preserve alignment
between article IDs and vectors.

In [ ]:
summary = []
for name in (*ARTIFACTS, MANIFEST_NAME):
    shape = (
        f"{N} x {EMBEDDING_DIM}"
        if name == "text_embeddings.npy"
        else str(N)
        if name != MANIFEST_NAME
        else "manifest"
    )
    summary.append(
        {
            "path": str(OUTPUT_DIR / name),
            "bytes": (OUTPUT_DIR / name).stat().st_size,
            "shape_or_rows": shape,
            "status": "validated",
        }
    )
display(pa.Table.from_pylist(summary).to_pandas())
print(
    {
        "reused": REUSED,
        "recorded_encoding_measurements": manifest["measurements"],
    }
)

In [ ]:
reloaded_embeddings = np.load(
    RUN_DIR / "text_embeddings.npy", mmap_mode="r", allow_pickle=False
)
reloaded_mapping = pq.read_table(RUN_DIR / "text_article_ids.parquet")
reloaded_index = faiss.read_index(str(RUN_DIR / "text_faiss.index"))
print(validate_artifacts(RUN_DIR, article_ids, texts, max_seq_length))

### Conclusions and Next Steps

A completed run provides one validated static text vector per source article,
with an auditable mapping and immutable model revision. Artifact validation
does not establish semantic quality or downstream recommendation accuracy.

Real L4 execution, CUDA out-of-memory recovery and Drive publication remain
unverified by local synthetic tests. After an authorized Colab run, inspect the
token audit, output table and manifest measurements before using the artifacts.
Keep those observations separate from the configuration assumptions above.